In [ ]:
"""
save_data_label.py에서 만든 엑셀파일을 이용해서 키포인트 데이터와 라벨을 저장하는 코드입니다.
각 키포인트는 2차원 좌표로 표현되며, 각 좌표는 (x, y) 형태입니다.
라벨은 각 데이터에 대한 인덱스값으로 저장됩니다.
정규화해 저장합니다
dataXX-norm 폴더에 저장됩니다

데이터 구조:
"keypoint"
	•	왼손 키포인트 21개 × 2 = 42개
	•	오른손 키포인트 21개 × 2 = 42개
	•	총 84
"label"
    •	라벨의 인덱스값
"""

'\n"keypoint"\n\t•\t왼손 키포인트 21개 × 2 = 42개\n\t•\t오른손 키포인트 21개 × 2 = 42개\n\t•\t총 84\n"label"\n    •\t라벨의 인덱스값\n'

In [8]:
import os
import json
import numpy as np
import pandas as pd
import math


In [ ]:
# input : mediapipe 손 keypoint (21,2)
# out : 정규화
def norm(x):
    mean = x.mean(axis=0, keepdims=True)
    std  = x.std(axis=0, keepdims=True) + 1e-6
    x = (x - mean) / std
    return x

In [16]:

def extract_start_end_from_filename(folder_path, filename):
    # JSON 파일 경로
    json_path = os.path.join(folder_path, filename + "_morpheme.json") # NIA_SL_WORD0001_REAL01_D_morpheme.json

    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # data 필드 안의 리스트에서 start와 end 추출
    for segment in data['data']:
        start_time = segment['start']
        end_time = segment['end']
        print(f"start: {start_time}, end: {end_time}")
    return start_time, end_time


def load_keypoints_from_folder(folder_path, filename_prefix, start_sec=None, end_sec=None, fps=30):
    keypoints_list = []
    files = sorted([
        f for f in os.listdir(folder_path)
        if f.startswith(filename_prefix) and f.endswith('_keypoints.json')
    ])

    # start_sec, end_sec를 프레임 번호로 변환
    if start_sec is not None and end_sec is not None:
        start_frame = math.floor(start_sec * fps)
        end_frame = math.ceil(end_sec * fps)
    else:
        start_frame = 0
        end_frame = len(files) - 1

    # 파일명에서 프레임 번호 추출하는 함수
    def extract_frame_number(filename):
        # 예: NIA_SL_WORD0001_REAL01_D_000000000052_keypoints.json
        # 숫자만 추출해서 int로 변환
        import re
        match = re.search(r'_(\d{12})_keypoints.json', filename)
        if match:
            return int(match.group(1))
        return -1
    
    def extract_xy(keypoints_2d):
        keypoints_2d = np.array(keypoints_2d)
        if len(keypoints_2d) % 3 != 0:
            # 데이터가 이상할 때 대비
            return np.array([])
        xy = keypoints_2d.reshape(-1, 3)[:, :2]
        return xy

    for file in files:
        frame_num = extract_frame_number(file)
        if frame_num == -1:
            continue

        # start_frame과 end_frame 사이만 처리
        if frame_num < start_frame or frame_num > end_frame:
            continue

        full_path = os.path.join(folder_path, file)
        with open(full_path, 'r') as f:
            data = json.load(f)
            people = data.get("people", {})
            if not people:
                keypoints_list.append(np.zeros(0))
                continue

            # pose = people.get("pose_keypoints_2d", [])
            # face = people.get("face_keypoints_2d", [])
            # frame_keypoints = pose + hand_left + hand_right + face

            hand_left = people.get("hand_left_keypoints_2d", [])
            hand_right = people.get("hand_right_keypoints_2d", [])

            hand_left_xy = extract_xy(hand_left)
            hand_right_xy = extract_xy(hand_right)

            hand_left_x = norm(hand_left_xy[:, 0])
            hand_left_y = norm(hand_left_xy[:, 1])
            hand_right_x = norm(hand_right_xy[:, 0])
            hand_right_y = norm(hand_right_xy[:, 1])
            hand_left_xy = np.concatenate([hand_left_x, hand_left_y], axis=0)
            hand_right_xy = np.concatenate([hand_right_x, hand_right_y], axis=0)

            # left_angle = cal_hand_angle(hand_left_xy)
            # right_angle = cal_hand_angle(hand_right_xy)
            
            frame_keypoints =  np.concatenate([hand_left_xy.flatten(), hand_right_xy.flatten()], axis=0)
            # 21개 * 2 + 21 * 2 + 15 + 15 = 114
            keypoints_list.append(frame_keypoints)

    return np.array(keypoints_list)

def save_npz_for_video(keypoints_array, label_index, save_path):
    np.savez_compressed(save_path, keypoints=keypoints_array, label=label_index)

# data 순서
number = "10"

# 엑셀파일 경로
excel_path = f'light_data_label/sign_language_labels_{number}.xlsx'

# 키포인트 json 파일들이 있는 최상위 폴더 경로 (예: '/Users/hyeonji/Downloads/수어 영상/1.Training/01/')
base_folder = f'/Users/hyeonji/Downloads/수어 영상/1.Training/{number}'
morpheme_folder = f'/Users/hyeonji/Downloads/수어 영상/1.Training/morpheme/{number}'

save_folder = f'data{number}-norm'

# 엑셀 읽기
df = pd.read_excel(excel_path)

for idx, row in df.iterrows():
    video_filename = row['file_name']  # ex: 'NIA_SL_WORD1119_REAL01_R.mp4'
    label_index = row['label_index']   # ex: 2301

    # 확장자(.mp4) 제거해서 폴더명 or 파일명 prefix 만들기
    prefix = os.path.splitext(video_filename)[0]

    folder_path = os.path.join(base_folder, prefix)
    if not os.path.exists(folder_path):
        print(f"[경고] 폴더가 없습니다: {folder_path}")
        continue

    print(f"처리 중: {prefix} (라벨: {label_index})")
    start, end = extract_start_end_from_filename(morpheme_folder, prefix)
    keypoints_array = load_keypoints_from_folder(folder_path, prefix, start, end)
    save_path = os.path.join(save_folder, prefix + '.npz')
    save_npz_for_video(keypoints_array, label_index, save_path)

    print(f"저장 완료: {save_path}")

print("모든 영상 처리 완료.")

처리 중: NIA_SL_WORD2205_REAL10_F (라벨: 96)
start: 1.727, end: 3.431
저장 완료: data10-norm/NIA_SL_WORD2205_REAL10_F.npz
처리 중: NIA_SL_WORD0288_REAL10_F (라벨: 116)
start: 1.493, end: 3.611
저장 완료: data10-norm/NIA_SL_WORD0288_REAL10_F.npz
처리 중: NIA_SL_WORD1409_REAL10_F (라벨: 91)
start: 1.407, end: 3.108
저장 완료: data10-norm/NIA_SL_WORD1409_REAL10_F.npz
처리 중: NIA_SL_WORD1251_REAL10_F (라벨: 81)
start: 1.751, end: 2.807
저장 완료: data10-norm/NIA_SL_WORD1251_REAL10_F.npz
처리 중: NIA_SL_WORD0126_REAL10_F (라벨: 102)
start: 1.382, end: 3.098
저장 완료: data10-norm/NIA_SL_WORD0126_REAL10_F.npz
처리 중: NIA_SL_WORD2524_REAL10_F (라벨: 113)
start: 2.231, end: 4.201
저장 완료: data10-norm/NIA_SL_WORD2524_REAL10_F.npz
처리 중: NIA_SL_WORD0301_REAL10_F (라벨: 33)
start: 1.746, end: 3.177
저장 완료: data10-norm/NIA_SL_WORD0301_REAL10_F.npz
처리 중: NIA_SL_WORD0094_REAL10_F (라벨: 28)
start: 1.351, end: 3.176
저장 완료: data10-norm/NIA_SL_WORD0094_REAL10_F.npz
처리 중: NIA_SL_WORD2347_REAL10_F (라벨: 32)
start: 1.642, end: 3.48
저장 완료: data10-norm/NIA_SL_WOR